# 21.1 流式音视频交互管线

状态机 + barge-in + 分段延迟预算仿真。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
from enum import Enum, auto


class State(Enum):
    IDLE = auto(); LISTENING = auto(); ASR = auto(); LLM = auto(); TTS = auto(); SPEAKING = auto()


class VoicePipeline:
    def __init__(self):
        self.state = State.IDLE
        self.log = []

    def _go(self, st):
        self.state = st; self.log.append(st.name)

    def on_vad_speech(self):
        if self.state in (State.IDLE, State.SPEAKING, State.TTS):
            # barge-in：打断播报
            self._go(State.LISTENING)

    def on_vad_end(self):
        if self.state == State.LISTENING:
            self._go(State.ASR)

    def on_asr_final(self):
        if self.state == State.ASR:
            self._go(State.LLM)

    def on_llm_first_sentence(self):
        if self.state == State.LLM:
            self._go(State.TTS)

    def on_tts_start(self):
        if self.state == State.TTS:
            self._go(State.SPEAKING)

    def on_tts_end(self):
        if self.state == State.SPEAKING:
            self._go(State.IDLE)


p = VoicePipeline()
for ev in ["on_vad_speech","on_vad_end","on_asr_final","on_llm_first_sentence","on_tts_start","on_vad_speech","on_vad_end"]:
    getattr(p, ev)()
print(" -> ".join(p.log))

In [ ]:
BUDGET_MS = {"vad_tail": 120, "asr_final": 220, "llm_ttft": 180, "tts_first": 140}


def e2e_first_audio(profile):
    return sum(profile[k] for k in ("vad_tail","asr_final","llm_ttft","tts_first"))


prof = dict(BUDGET_MS)
print("E2E first audio", e2e_first_audio(prof), "ms; target <800:", e2e_first_audio(prof)<800)
# 半双工：SPEAKING 时不做 ASR；全双工需要 AEC
print("half-duplex simpler on mobile; full-duplex needs AEC + barge-in policy")

## 小结

体验助手体验由状态机与预算决定；打断是一等公民；车载免提必须考虑 AEC。